# Chronic Kidney Disease (CKD) Dataset Exploration
## Bonus Question: 1D Deep Convolutional Neural Network (CNN) Classification

### Overview & Objective
This notebook performs comprehensive exploratory data analysis (EDA) on the **Chronic Kidney Disease (CKD)** dataset in preparation for developing a **1D Convolutional Neural Network (CNN)** for tabular medical classification.

The primary goals of this exploration are:
1. Verify the Python runtime and library environment.
2. Robustly load the dataset directly from its native **ARFF (Attribute-Relation File Format)** representation using relative paths.
3. Inspect structural dimensions, feature names, and raw data representations.
4. Perform rigorous missing-value analysis (identifying `'?'` tokens, nulls, and frequency per column).
5. Identify and classify all 24 predictive features into numerical and nominal/categorical types, and inspect the binary target class (`'ckd'` vs `'notckd'`).
6. Examine descriptive statistics, value distributions, anomalies, and target class balance.
7. Generate meaningful exploratory visualizations (missingness patterns, class distribution, numerical histograms/boxplots, and categorical distributions).
8. Summarize key observations and critical implications for downstream preprocessing and 1D CNN architecture design.

## 1. Imports and Environment Verification
We verify that all essential libraries (`numpy`, `pandas`, `matplotlib`, `seaborn`) are properly imported and record their versions for reproducibility.

In [ ]:
import sys
import os
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization aesthetics
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 10

print("=" * 55)
print("Environment Verification:")
print(f"  Operating System : {platform.system()} {platform.release()} ({platform.machine()})")
print(f"  Python Version   : {sys.version.split()[0]}")
print(f"  NumPy Version    : {np.__version__}")
print(f"  Pandas Version   : {pd.__version__}")
print(f"  Matplotlib Version: {plt.matplotlib.__version__}")
print(f"  Seaborn Version  : {sns.__version__}")
print("=" * 55)

## 2. Loading the ARFF Dataset

### ARFF Parser Implementation
Standard CSV loaders (`pd.read_csv`) fail on ARFF files due to metadata directives (`@relation`, `@attribute`, `@data`) and comment lines (`%`). Furthermore, the raw CKD dataset contains minor whitespace/tab inconsistencies and trailing delimiters in a few rows.

To ensure robust, dependency-free execution compatible with the existing classification pipeline (`classification/results/classification_ (2).py`), we define an ARFF reader function `load_arff_dataset`.

### File Selection: `chronic_kidney_disease.arff` vs `chronic_kidney_disease_full.arff`
Both files contain identical 400 patient data instances and 25 attributes. `chronic_kidney_disease_full.arff` contains embedded docstrings/comments in the header, while `chronic_kidney_disease.arff` is the primary streamlined ARFF file utilized by the existing benchmark classification models in this repository. We configure the loader with relative pathing from `classification/04_deep_cnn/` to `../../datasets/raw/chronic_kidney_disease/`.

In [ ]:
def load_arff_dataset(file_path):
    """
    Parse an ARFF file into attribute specifications and a pandas DataFrame.
    
    Handles:
    - Comment lines starting with '%'
    - Relation and attribute metadata declarations
    - Whitespace stripping and trailing comma tokens in data records
    """
    attributes = []
    data_rows = []
    in_data_section = False
    
    if not os.path.isfile(file_path):
        raise FileNotFoundError(f"ARFF file not found at: {os.path.abspath(file_path)}")
        
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            stripped = line.strip()
            if not stripped or stripped.startswith("%"):
                continue
            
            if stripped.lower().startswith("@attribute"):
                parts = stripped.split()
                # Attribute name is token 1, type/domain definition is remainder
                attr_name = parts[1].strip("'\"")
                attr_def = " ".join(parts[2:])
                attributes.append((attr_name, attr_def))
                
            elif stripped.lower().startswith("@data"):
                in_data_section = True
                
            elif in_data_section:
                # Split CSV data line
                vals = [v.strip() for v in stripped.split(",")]
                
                # Clean trailing empty strings caused by trailing commas
                while len(vals) > len(attributes) and vals[-1] == "":
                    vals.pop()
                if len(vals) > len(attributes) and "" in vals:
                    vals.remove("")
                    
                data_rows.append(vals)
                
    col_names = [attr[0] for attr in attributes]
    df = pd.DataFrame(data_rows, columns=col_names)
    
    # Strip any residual whitespace from string values
    for col in df.columns:
        df[col] = df[col].astype(str).str.strip()
        
    return df, attributes

# Relative path from classification/04_deep_cnn/ to datasets/raw/chronic_kidney_disease/
arff_path = os.path.join("..", "..", "datasets", "raw", "chronic_kidney_disease", "chronic_kidney_disease.arff")

df_raw, attribute_metadata = load_arff_dataset(arff_path)
print(f"Successfully loaded ARFF file from: {arff_path}")
print(f"Total attributes declared in header : {len(attribute_metadata)}")
print(f"DataFrame shape                    : {df_raw.shape}")

## 3. Dataset Shape and Attribute Names
Let us inspect the dimensions and list all 25 attribute names (24 feature inputs + 1 target class).

In [ ]:
print(f"Dataset Dimensions: {df_raw.shape[0]} rows (records) x {df_raw.shape[1]} columns (attributes)\n")
print("Declared Attributes & Schema Types:")
for idx, (name, atype) in enumerate(attribute_metadata, start=1):
    print(f"  {idx:2d}. {name:<8} : {atype}")

## 4. Initial Inspection and Data Representation
Because ARFF stores values as text tokens, missing values are denoted by `'?'`. When loaded into pandas prior to conversion, all columns initially have `object` data type.

Below, we view the raw head of the dataset and verify column-level types.

In [ ]:
print("Raw column types before parsing:")
print(df_raw.dtypes)
print("\nFirst 5 records (raw string tokens including '?' missing indicators):")
display(df_raw.head())

## 5. Missing-Value Analysis
In the CKD dataset documentation (`chronic_kidney_disease.info.txt`), missing values are denoted by `'?'`.

We now:
1. Replace `'?'`, empty strings `""`, and `'nan'` tokens with proper `np.nan` values.
2. Count missing values per attribute and calculate percentage missingness.
3. Analyze sample-level missingness (how many complete records exist).

In [ ]:
# Create a working copy with true NaN values for exploration
df = df_raw.replace({"?": np.nan, "": np.nan, "nan": np.nan}).copy()

# Tabulate missing values
missing_counts = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    "Attribute": df.columns,
    "Missing Count": missing_counts,
    "Missing Percentage (%)": missing_pct.round(2)
}).sort_values(by="Missing Count", ascending=False).reset_index(drop=True)

print(f"Total missing entries across entire dataset : {missing_counts.sum()} out of {df.size} ({missing_counts.sum() / df.size * 100:.2f}%)")
print(f"Attributes with missing values              : {(missing_counts > 0).sum()} / {df.shape[1]}")
print(f"Complete cases (records with no missing values): {(df.isnull().sum(axis=1) == 0).sum()} / {len(df)} ({(df.isnull().sum(axis=1) == 0).sum() / len(df) * 100:.2f}%)")

print("\nMissing Values Summary (Top 15 attributes with missing data):")
display(missing_df.head(15))

## 6. Numerical and Categorical Feature Identification
Based on the official clinical dataset documentation and attribute declarations:
- **Numerical Features (14 attributes)**:
  - `age`: Age in years
  - `bp`: Blood Pressure (mm/Hg)
  - `sg`: Specific Gravity (clinical urinary measurement; ordinal nominal in ARFF, numerical ratio scale in physiological analysis)
  - `al`: Albumin (0 to 5 urinary protein scale)
  - `su`: Sugar (0 to 5 urinary glucose scale)
  - `bgr`: Blood Glucose Random (mgs/dl)
  - `bu`: Blood Urea (mgs/dl)
  - `sc`: Serum Creatinine (mgs/dl)
  - `sod`: Sodium (mEq/L)
  - `pot`: Potassium (mEq/L)
  - `hemo`: Hemoglobin (gms)
  - `pcv`: Packed Cell Volume (hematocrit %)
  - `wbcc`: White Blood Cell Count (cells/cumm; labeled wc in info.txt)
  - `rbcc`: Red Blood Cell Count (millions/cmm; labeled rc in info.txt)

- **Categorical Features (10 attributes)**:
  - `rbc`: Red Blood Cells (`normal`, `abnormal`)
  - `pc`: Pus Cell (`normal`, `abnormal`)
  - `pcc`: Pus Cell Clumps (`present`, `notpresent`)
  - `ba`: Bacteria (`present`, `notpresent`)
  - `htn`: Hypertension (`yes`, `no`)
  - `dm`: Diabetes Mellitus (`yes`, `no`)
  - `cad`: Coronary Artery Disease (`yes`, `no`)
  - `appet`: Appetite (`good`, `poor`)
  - `pe`: Pedal Edema (`yes`, `no`)
  - `ane`: Anemia (`yes`, `no`)

- **Target Variable (1 attribute)**:
  - `class`: Chronic Kidney Disease status (`ckd` vs `notckd`)

We cast numerical features to `float64` for numerical exploration.

In [ ]:
numerical_columns = [
    "age", "bp", "sg", "al", "su",
    "bgr", "bu", "sc", "sod", "pot",
    "hemo", "pcv", "wbcc", "rbcc"
]

categorical_columns = [
    "rbc", "pc", "pcc", "ba",
    "htn", "dm", "cad", "appet",
    "pe", "ane"
]

target_column = "class"

print(f"Total features: {len(numerical_columns) + len(categorical_columns)} + 1 target = {len(df.columns)}")
print(f"  Numerical features ({len(numerical_columns)})   : {numerical_columns}")
print(f"  Categorical features ({len(categorical_columns)}) : {categorical_columns}")
print(f"  Target column                   : '{target_column}'")

# Convert numerical columns to float
for col in numerical_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("\nVerified Data Types After Conversion:")
print(df.dtypes)

## 7. Target Class Distribution
We examine the distribution of the target variable `class` to understand class balance.

In [ ]:
target_counts = df[target_column].value_counts(dropna=False)
target_proportions = df[target_column].value_counts(normalize=True, dropna=False) * 100

target_summary = pd.DataFrame({
    "Class": target_counts.index,
    "Count": target_counts.values,
    "Percentage (%)": target_proportions.values.round(2)
})

print("Target Class Distribution:")
display(target_summary)
print(f"Class ratio (CKD : Non-CKD): {target_counts['ckd'] / target_counts['notckd']:.2f} : 1")

## 8. Descriptive Statistics
We examine parametric and non-parametric summary statistics for all numerical features, followed by category frequencies for nominal features.

In [ ]:
print("Numerical Features Descriptive Statistics:")
display(df[numerical_columns].describe().T.round(2))

In [ ]:
print("Categorical Features Summary & Value Frequencies:")
cat_summary = []
for col in categorical_columns:
    val_counts = df[col].value_counts(dropna=False).to_dict()
    cat_summary.append({
        "Feature": col,
        "Unique Categories": df[col].dropna().nunique(),
        "Category Breakdown": val_counts,
        "Missing": df[col].isnull().sum()
    })

display(pd.DataFrame(cat_summary))

## 9. Exploratory Visualizations
We generate insightful visual diagnostics:
1. Missing-value frequency across all features.
2. Target class balance.
3. Numerical feature distributions and outlier boxplots.
4. Categorical feature distributions grouped by target class.

In [ ]:
# 1. Missing Values Bar Chart
plt.figure(figsize=(12, 5))
missing_sorted = missing_df[missing_df["Missing Count"] > 0].sort_values(by="Missing Count", ascending=True)
bars = plt.barh(missing_sorted["Attribute"], missing_sorted["Missing Count"], color="#4A90E2", edgecolor="black", alpha=0.8)
plt.xlabel("Number of Missing Records (out of 400 total)")
plt.ylabel("Attribute")
plt.title("Missing Values Count by Attribute (CKD Dataset)", fontsize=13, fontweight="bold")
plt.grid(axis="x", linestyle="--", alpha=0.5)

# Annotate bars with counts
for bar in bars:
    w = bar.get_width()
    plt.text(w + 2, bar.get_y() + bar.get_height() / 2, f"{int(w)} ({w/len(df)*100:.1f}%)", 
             va="center", fontsize=9)

plt.xlim(0, 180)
plt.tight_layout()
plt.show()

In [ ]:
# 2. Target Class Distribution Plot
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Countplot
sns.countplot(data=df, x="class", ax=axes[0], palette=["#E74C3C", "#2ECC71"], order=["ckd", "notckd"])
axes[0].set_title("Target Count: Chronic Kidney Disease", fontweight="bold")
axes[0].set_xlabel("Class")
axes[0].set_ylabel("Number of Patients")
for p in axes[0].patches:
    axes[0].annotate(f"{int(p.get_height())} ({p.get_height()/len(df)*100:.1f}%)",
                     (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                     ha="center", va="center", color="white", fontweight="bold", fontsize=11)

# Pie chart
axes[1].pie(target_counts, labels=["CKD (1)", "Not CKD (0)"], autopct="%1.1f%%",
            startangle=140, colors=["#E74C3C", "#2ECC71"], explode=(0.04, 0), shadow=True)
axes[1].set_title("Target Class Proportions", fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# 3. Numerical Feature Histograms with KDE stratified by Class
fig, axes = plt.subplots(5, 3, figsize=(15, 16))
axes = axes.flatten()

for idx, col in enumerate(numerical_columns):
    sns.histplot(data=df, x=col, hue="class", kde=True, ax=axes[idx],
                 palette={"ckd": "#E74C3C", "notckd": "#2ECC71"}, alpha=0.5, element="step")
    axes[idx].set_title(f"Distribution of {col}", fontweight="bold", fontsize=10)
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel("Count")

# Remove unused 15th subplot
fig.delaxes(axes[14])

plt.suptitle("Distributions of Numerical Features by Target Class (CKD vs Not CKD)", fontsize=14, fontweight="bold", y=1.002)
plt.tight_layout()
plt.show()

In [ ]:
# 4. Boxplots for Numerical Features (Identifying Outliers & Group Discrepancies)
fig, axes = plt.subplots(5, 3, figsize=(15, 16))
axes = axes.flatten()

for idx, col in enumerate(numerical_columns):
    sns.boxplot(data=df, x="class", y=col, ax=axes[idx],
                palette={"ckd": "#E74C3C", "notckd": "#2ECC71"})
    axes[idx].set_title(f"Boxplot of {col} by Class", fontweight="bold", fontsize=10)
    axes[idx].set_xlabel("Class")
    axes[idx].set_ylabel(col)

fig.delaxes(axes[14])

plt.suptitle("Boxplots of Numerical Features Across Classes", fontsize=14, fontweight="bold", y=1.002)
plt.tight_layout()
plt.show()

In [ ]:
# 5. Categorical Feature Distributions by Target Class
fig, axes = plt.subplots(5, 2, figsize=(13, 16))
axes = axes.flatten()

for idx, col in enumerate(categorical_columns):
    sns.countplot(data=df, x=col, hue="class", ax=axes[idx],
                  palette={"ckd": "#E74C3C", "notckd": "#2ECC71"})
    axes[idx].set_title(f"Prevalence of {col} by Class", fontweight="bold", fontsize=10)
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel("Count")
    axes[idx].legend(title="Class", loc="upper right")

plt.suptitle("Categorical Predictors Stratified by Target Class", fontsize=14, fontweight="bold", y=1.002)
plt.tight_layout()
plt.show()

## 10. Numerical Feature Correlation Analysis
We compute Spearman rank correlation among numerical features to evaluate inter-feature multicollinearity and clinical associations without assuming normality.

In [ ]:
plt.figure(figsize=(11, 8))
corr_matrix = df[numerical_columns].corr(method="spearman")
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            vmin=-1, vmax=1, cbar_kws={"shrink": 0.8}, linewidths=0.5)
plt.title("Spearman Rank Correlation Heatmap (Numerical Features)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 11. Summary of Exploratory Observations & CNN Implications

### 1. Dataset Dimensions & Schema
- The dataset comprises **400 patient records** and **25 attributes** (24 diagnostic features + 1 target class).
- Features consist of **14 numerical attributes** (`age`, `bp`, `sg`, `al`, `su`, `bgr`, `bu`, `sc`, `sod`, `pot`, `hemo`, `pcv`, `wbcc`, `rbcc`) and **10 categorical attributes** (`rbc`, `pc`, `pcc`, `ba`, `htn`, `dm`, `cad`, `appet`, `pe`, `ane`).

### 2. Missing Values Analysis
- Total missing entries: **1,012 missing values** across the dataset (~10.12% of all table cells).
- Only **158 out of 400 records (39.5%)** are complete cases. Dropping missing rows is not viable as it would discard over 60% of patient data.
- Attributes with the highest missingness:
  - `rbc` (red blood cells): 152 missing (38.0%)
  - `rbcc` (red blood cell count): 131 missing (32.8%)
  - `wbcc` (white blood cell count): 106 missing (26.5%)
  - `pot` & `sod` (electrolytes): ~87-88 missing (~22%)
  - `pcv` (packed cell volume): 71 missing (17.75%)
  - `pc` (pus cell): 65 missing (16.25%)
- *Implication:* A systematic imputation strategy (e.g., median for numerical features and most-frequent mode for categorical features) is essential prior to training.

### 3. Target Class Distribution
- `ckd` (positive class): **250 instances (62.5%)**
- `notckd` (negative class): **150 instances (37.5%)**
- The target displays mild class imbalance (1.67:1 ratio). Stratified sampling and threshold-insensitive evaluation metrics (such as ROC-AUC, F1-score, and Recall) must be preserved in downstream modeling.

### 4. Key Clinical & Statistical Insights
- **Strong Discriminators**:
  - `hemo` (hemoglobin), `pcv` (packed cell volume), and `rbcc` (red blood cell count) show distinct separation: CKD patients have substantially lower levels due to impaired erythropoietin production.
  - `sc` (serum creatinine) and `bu` (blood urea) are heavily right-skewed and elevated in CKD patients.
  - Among categorical variables, `htn` (hypertension) and `dm` (diabetes mellitus) are strong co-morbidities for CKD.
- **Multicollinearity**:
  - `hemo`, `pcv`, and `rbcc` are strongly correlated ($r > 0.85$), reflecting linked hematological markers.

### 5. Architectural Implications for 1D CNN Implementation
1. **Tabular Nature (Not 2D Image)**: The data has 24 heterogeneous features rather than spatial grid coordinates.
2. **Channel Representation for 1D CNN**:
   - Tabular features after one-hot encoding categorical variables and standardizing numerical variables will produce a feature vector of length $D$.
   - For a 1D Convolutional Neural Network (`Conv1D`), this vector can be represented as an input tensor of shape `(batch_size, length=D, channels=1)` (or `(batch_size, 1, D)` depending on framework layout).
3. **Sequence Order & Convolutional Filter Size**:
   - Unlike time-series or speech signals, adjacent columns in a tabular dataset do not inherently possess local temporal or spatial autocorrelation.
   - Small kernel sizes ($k=3$ or $k=2$) with appropriate padding, along with residual connections or dropout, will help prevent overfitting on this small sample size ($N=400$, with train set $N_{train}=320$).
4. **Leakage Prevention**: All imputation and scaling parameters must strictly be fit on the training split only and transformed on the validation/test splits.